# Model

- $t_i$: Survival time
- $d_i$: Event indicator (1 is death, 0 is censored)
- $x_i \in \mathbb{R}^p$, where $p=13,859$: Gene expression vector
- $s(i)$ site label

## Weibull AFT model

$$
\log(t_i) = \mu + \beta \cdot x_i + \varepsilon_i
$$

- $\mu$ is a global intercept (the baseline log survival time)
- $\boldsymbol{\beta} \in \mathbb{R}^p$ is a vector of gene coefficients
- $\varepsilon_i​$ is a residual error term

And
$$
\varepsilon_i \in \text{Gumbel}
$$


Equivalently
$$
t_i \sim \text{Weibull}(\alpha, \lambda_i)
$$

where $\alpha$ is the shape parameter and $\lambda_i = \exp(\mu + \beta\cdot x_i)$

## Handling censored data

If the patient died at time $t$, then that is the approximate probability density function of time of death.

If the patient is still alive by time $t$, then that is the survival function.

So
$$
\mathcal{L}_i = \begin{cases}
f(t_i) & d_i = 1 \text{ (died)} \\
S(t_i) & d_i = 0 \text{ (censored)}
\end{cases}
$$

where $f$ is the Weibull probability density function and $S$ is the survival function 

## Priors

The number of covariates (13,859) is much bigger than the number of data points (495). But most genes are irrelevant, only a few have an effect. So we use an **horseshoe prior**

For gene $j$, we have

$$
\beta_j \sim \mathcal{N}(0, \tau^2\lambda_j^2)
$$

$$
\lambda_j \sim \text{Half-Cauchy}(1)
$$

$$
\tau \sim \text{Half-Student-t}(3, 0, \sigma_\tau)
$$

Here
- $\lambda_j$ is the local shrinkage for gene $j$
- $\tau$ is the global shrinkage 

The $\sigma_\tau$ is calibrated using Piironen & Vehtari formula:
$$
\sigma_\tau = \frac{p_0}{p-p_0}\cdot \frac{\sigma}{\sqrt{n}}
$$

where
- $p_0$ is the prior guess for the number of relevant genes
- $p$ is the total number of genes
- $n$ is the number of patients
- $\sigma$ is the residual

### Model A

$$
\log (t_i) = \mu + \sum_{j=1}^p \beta_j x_{ij} + \varepsilon_i
$$

where $\beta_j$ follows the horshoe prior


In [ ]:
import pymc as pm
import numpy as np
import pandas as pd
import pytensor.tensor as pt

def build_model_a(df: pd.DataFrame, gene_cols: list, p0: int = 30,
                  time_col: str = "OS_time", event_col: str = "OS",
                  slab_scale: float = 2.0, slab_df: float = 4.0) -> pm.Model:
    """
    Model A — Naive Weibull AFT with REGULARIZED horseshoe prior on gene
    coefficients (Piironen & Vehtari 2017). No site adjustment. Handles
    right-censoring. The slab (c2) caps large coefficients so the linear
    predictor can't blow up the Weibull likelihood -> avoids divergences.
    Endpoint is parametrized: default OS; pass time_col/event_col for PFI etc.
    """
    # survival models require t > 0 and a non-missing event/time
    df = df[(df[time_col] > 0) & df[event_col].notna() & df[time_col].notna()]

    X = df[gene_cols].values
    t = df[time_col].values
    d = df[event_col].values.astype(int)

    n, p = X.shape
    obs_mask, cens_mask = d == 1, d == 0
    X_obs, X_cens = X[obs_mask], X[cens_mask]
    t_obs, t_cens = t[obs_mask], t[cens_mask]

    # horseshoe global-scale calibration (Piironen & Vehtari)
    sigma_est = np.std(np.log1p(t))
    sigma_tau = (p0 / (p - p0)) * (sigma_est / np.sqrt(n))

    with pm.Model() as model:
        mu    = pm.Normal("mu", mu=6, sigma=2)
        alpha = pm.HalfNormal("alpha", sigma=1)

        # regularized horseshoe (non-centered)
        tau = pm.HalfStudentT("tau", nu=3, sigma=sigma_tau)        # global shrinkage
        lam = pm.HalfCauchy("lam", beta=1, shape=p)                # local shrinkage
        c2  = pm.InverseGamma("c2", alpha=slab_df / 2,             # slab (caps big coefs)
                              beta=(slab_df / 2) * slab_scale**2)
        lam_tilde = pt.sqrt(c2 * lam**2 / (c2 + tau**2 * lam**2))
        z    = pm.Normal("z", mu=0, sigma=1, shape=p)
        beta = pm.Deterministic("beta", z * tau * lam_tilde)

        eta_obs  = mu + pm.math.dot(X_obs,  beta)
        eta_cens = mu + pm.math.dot(X_cens, beta)

        # Weibull AFT: scale lambda = exp(eta)
        pm.Weibull("y_obs", alpha=alpha, beta=pm.math.exp(eta_obs), observed=t_obs)
        # censored: log S(t) = -(t/lambda)^alpha = -exp(alpha*(log t - eta))  [stable, t>0]
        pm.Potential("cens_ll", (-pm.math.exp(alpha * (np.log(t_cens) - eta_cens))).sum())

    return model

In [ ]:
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import gseapy as gp

# per-cohort I/O: everything for this cohort lives under data/processed/{COHORT}/
COHORT = "LUSC"
OUT = Path(f"../data/processed/{COHORT}")
(OUT / "traces").mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(OUT / f"{COHORT}.parquet")

# --- MSigDB Hallmark panel -> Ensembl -> intersect with matrix ---
hm = gp.get_library(name="MSigDB_Hallmark_2020", organism="Human")   # 50 sets
symbols = {g for genes in hm.values() for g in genes}

# symbol -> versioned ENSG from any STAR annotation TSV
tsv = glob.glob(f"../data/raw/{COHORT}/*/*.rna_seq.augmented_star_gene_counts.tsv")[0]
ref = pd.read_csv(tsv, sep="\t", skiprows=1)
ref = ref[ref["gene_id"].str.startswith("ENSG")]
sym2ens = dict(zip(ref["gene_name"], ref["gene_id"]))
hm_ens = {sym2ens[s] for s in symbols if s in sym2ens}

gene_cols = [c for c in df.columns if c.startswith("ENSG") and c in hm_ens]
print(f"hallmark genes used: {len(gene_cols)}  (of {len(symbols)} symbols)")
print(f"Patients: {len(df)}  Deaths: {int(df['OS'].sum())}  Censored: {int((df['OS']==0).sum())}")

model = build_model_a(df, gene_cols, p0=10)

with model:
    trace = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.99,             # 0.99 to clear the funnel-tail divergences
        nuts_sampler="nutpie",          # Rust NUTS: faster, handles its own parallelism
        return_inferencedata=True,
    )

print(az.summary(trace, var_names=["mu", "alpha", "tau"], round_to=3))
print(f"Divergences: {int(trace.sample_stats.diverging.sum())}")

### Model A — prognostic gene hits

With strong global shrinkage (`tau` ≈ 0) most coefficients sit at ~0. Genes whose 89% ETI excludes 0 are the candidate prognostic signals. If *none* clear the bar, that is itself a finding (no individual hallmark gene survives shrinkage), not a bug.

In [ ]:
# --- Model A: which hallmark genes carry a prognostic signal? ---
# beta[i] aligns with gene_cols[i]. A gene "hits" if its 89% ETI excludes 0.
ens2sym = {v: k for k, v in sym2ens.items()}

bsa = az.summary(trace, var_names=["beta"]).reset_index(drop=True)
# arviz 1.x returns object-dtype columns -> coerce before comparing
num_cols = ["mean", "sd", "eti89_lb", "eti89_ub"]
bsa[num_cols] = bsa[num_cols].apply(pd.to_numeric, errors="coerce")
bsa["gene_id"] = gene_cols
bsa["symbol"]  = [ens2sym.get(g, g) for g in gene_cols]

hits_a = bsa[(bsa["eti89_lb"] > 0) | (bsa["eti89_ub"] < 0)].copy()
hits_a = hits_a.reindex(hits_a["mean"].abs().sort_values(ascending=False).index)

print(f"Model A: {len(hits_a)} / {len(gene_cols)} genes with 89% ETI excluding 0")
print("positive beta = longer survival (AFT scale), negative = shorter")
hits_a[["symbol", "gene_id", "mean", "sd", "eti89_lb", "eti89_ub"]].head(25)

In [ ]:
# save the Model A trace so we don't have to re-sample
trace.to_netcdf(OUT / "traces" / "model_a_OS_hallmark.nc")
print("saved ->", OUT / "traces" / "model_a_OS_hallmark.nc")

### Model B - with site intercepts

$$
\log (t_i) = \mu + \sum_{j=1}^p \beta_j x_{ij} + \mu_{s(i)} + \varepsilon_i
$$

$$
\mu_s \sim \mathcal{N}(0, \sigma^2_\text{site}), \text s = 1\cdots S
$$

$$
\sigma_\text{site} \sim \text{Half-Normal}(1)
$$

In [ ]:
def build_model_b(df: pd.DataFrame, gene_cols: list, p0: int = 30,
                  time_col: str = "OS_time", event_col: str = "OS",
                  slab_scale: float = 2.0, slab_df: float = 4.0) -> pm.Model:
    """
    Model B — Weibull AFT + regularized horseshoe (identical to Model A) PLUS
    per-site (TSS) random intercepts mu_s. Endpoint parametrized (default OS).
    """
    df = df[(df[time_col] > 0) & df[event_col].notna() & df[time_col].notna()]

    X = df[gene_cols].values
    t = df[time_col].values
    d = df[event_col].values.astype(int)
    site_idx = pd.Categorical(df["tss"]).codes
    n_sites = int(site_idx.max() + 1)

    n, p = X.shape
    obs_mask, cens_mask = d == 1, d == 0
    X_obs, X_cens = X[obs_mask], X[cens_mask]
    t_obs, t_cens = t[obs_mask], t[cens_mask]
    s_obs, s_cens = site_idx[obs_mask], site_idx[cens_mask]

    sigma_est = np.std(np.log1p(t))
    sigma_tau = (p0 / (p - p0)) * (sigma_est / np.sqrt(n))

    with pm.Model() as model:
        mu    = pm.Normal("mu", mu=6, sigma=2)
        alpha = pm.HalfNormal("alpha", sigma=1)

        # site random intercepts (non-centered)
        sigma_site = pm.HalfNormal("sigma_site", sigma=1)
        z_site  = pm.Normal("z_site", mu=0, sigma=1, shape=n_sites)
        mu_site = pm.Deterministic("mu_site", z_site * sigma_site)

        # regularized horseshoe (non-centered) -- identical to Model A
        tau = pm.HalfStudentT("tau", nu=3, sigma=sigma_tau)
        lam = pm.HalfCauchy("lam", beta=1, shape=p)
        c2  = pm.InverseGamma("c2", alpha=slab_df / 2, beta=(slab_df / 2) * slab_scale**2)
        lam_tilde = pt.sqrt(c2 * lam**2 / (c2 + tau**2 * lam**2))
        z    = pm.Normal("z", mu=0, sigma=1, shape=p)
        beta = pm.Deterministic("beta", z * tau * lam_tilde)

        eta_obs  = mu + mu_site[s_obs]  + pm.math.dot(X_obs,  beta)
        eta_cens = mu + mu_site[s_cens] + pm.math.dot(X_cens, beta)

        pm.Weibull("y_obs", alpha=alpha, beta=pm.math.exp(eta_obs), observed=t_obs)
        pm.Potential("cens_ll", (-pm.math.exp(alpha * (np.log(t_cens) - eta_cens))).sum())

    return model

In [ ]:
model_b = build_model_b(df, gene_cols, p0=10)

with model_b:
    trace_b = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.99,
        nuts_sampler="nutpie",
        return_inferencedata=True,
    )

print(az.summary(trace_b, var_names=["mu", "alpha", "tau", "sigma_site"], round_to=3))
print(f"Divergences: {int(trace_b.sample_stats.diverging.sum())}")
trace_b.to_netcdf(OUT / "traces" / "model_b_OS_hallmark.nc")
print("saved ->", OUT / "traces" / "model_b_OS_hallmark.nc")

In [ ]:
# --- Model B: gene hits after site adjustment ---
bsb = az.summary(trace_b, var_names=["beta"]).reset_index(drop=True)
num_cols = ["mean", "sd", "eti89_lb", "eti89_ub"]
bsb[num_cols] = bsb[num_cols].apply(pd.to_numeric, errors="coerce")
bsb["gene_id"] = gene_cols
bsb["symbol"]  = [ens2sym.get(g, g) for g in gene_cols]

hits_b = bsb[(bsb["eti89_lb"] > 0) | (bsb["eti89_ub"] < 0)].copy()
hits_b = hits_b.reindex(hits_b["mean"].abs().sort_values(ascending=False).index)

print(f"Model B: {len(hits_b)} / {len(gene_cols)} genes with 89% ETI excluding 0")
hits_b[["symbol", "gene_id", "mean", "sd", "eti89_lb", "eti89_ub"]].head(25)

## Model A vs B — the survival-bias question

Model A ignores tissue source site; Model B adds site random intercepts $\mu_s$. If a gene's apparent signal in Model A was partly **confounded by site** (institutions that enrol sicker patients *and* differ systematically in expression), its coefficient should shrink toward 0 once we adjust for site. Genes surviving in **both** are robust to site confounding; genes appearing **only in A** are candidate artifacts of the bias — which is exactly what this study is about.

In [ ]:
# --- Does adjusting for tissue source site change the prognostic gene set? ---
set_a, set_b = set(hits_a["gene_id"]), set(hits_b["gene_id"])
print(f"Model A hits: {len(set_a)} | Model B hits: {len(set_b)}")
print(f"shared:                          {len(set_a & set_b)}")
print(f"A only (dropped after site adj): {len(set_a - set_b)}")
print(f"B only (gained after site adj):  {len(set_b - set_a)}")
print(f"site SD (sigma_site) mean:        {float(trace_b.posterior['sigma_site'].mean()):.3f}")

# coefficient shift for any gene flagged by either model
either = sorted(set_a | set_b)
if either:
    cmp = pd.DataFrame({
        "symbol": [ens2sym.get(g, g) for g in either],
        "beta_A": bsa.set_index("gene_id").loc[either, "mean"].values,
        "beta_B": bsb.set_index("gene_id").loc[either, "mean"].values,
    })
    cmp["abs_shrink_%"] = (1 - cmp["beta_B"].abs() / cmp["beta_A"].abs().replace(0, np.nan)) * 100
    display(cmp.reindex(cmp["beta_A"].abs().sort_values(ascending=False).index))
else:
    print("No hits in either model — nothing to compare.")

## RQ1 — Variance decomposition (ICC)

Partition the log-survival outcome variance from **Model B** into site, genomic, and residual components, computed **per posterior draw** (so we get a full posterior on each ICC, not a point estimate):

$$\sigma^2_\text{site} = \sigma_\text{site}^2 \qquad \sigma^2_\text{genomic} = \mathrm{Var}_i(\beta \cdot x_i) \qquad \sigma^2_\text{residual} = \frac{\pi^2}{6\alpha^2}$$

$$\text{ICC}_\text{site} = \frac{\sigma^2_\text{site}}{\sigma^2_\text{site} + \sigma^2_\text{genomic} + \sigma^2_\text{residual}}$$

`ICC_site` is the direct answer to **RQ1**: the share of survival-outcome variance attributable to tissue source site after controlling for genomics. Reported with an 89% ETI. (With `tau ≈ 0`, expect `σ²_genomic` to be small, so the split is mostly site vs. residual — that is itself the LUSC finding.)

In [ ]:
# --- RQ1: ICC variance decomposition from the Model B posterior ---
post = trace_b.posterior

# flatten posterior draws -> shape (..., S)
beta_s       = post["beta"].stack(sample=("chain", "draw")).values        # (p, S)
alpha_s      = post["alpha"].stack(sample=("chain", "draw")).values       # (S,)
sigma_site_s = post["sigma_site"].stack(sample=("chain", "draw")).values  # (S,)

# genomic linear-predictor variance across patients, per draw (same patients/genes as the fit)
Xb = df[df["OS_time"] > 0][gene_cols].values         # (n_patients, p)
eta_gen      = Xb @ beta_s                            # (n_patients, S)
var_genomic  = eta_gen.var(axis=0)                    # (S,)
var_site     = sigma_site_s ** 2                      # (S,)
var_residual = np.pi**2 / (6 * alpha_s**2)            # (S,)  Weibull/Gumbel residual variance

total = var_site + var_genomic + var_residual
icc = {
    "ICC_site":     var_site / total,
    "ICC_genomic":  var_genomic / total,
    "ICC_residual": var_residual / total,
}

rows = []
for name, x in icc.items():
    lo, hi = np.percentile(x, [5.5, 94.5])            # 89% ETI to match the rest of the notebook
    rows.append({"component": name, "mean": x.mean(), "eti89_lb": lo, "eti89_ub": hi})
icc_table = pd.DataFrame(rows)

print("variance components (posterior mean):")
print(f"  sigma_site = {sigma_site_s.mean():.3f}  -> var_site     = {var_site.mean():.4f}")
print(f"                              var_genomic  = {var_genomic.mean():.4f}")
print(f"  alpha      = {alpha_s.mean():.3f}  -> var_residual  = {var_residual.mean():.4f}")
print()
print(icc_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

In [ ]:
# --- p0 sensitivity: is "site >> genomics" robust to the shrinkage prior? ---
# p0 = prior guess for # relevant genes. Larger p0 -> weaker shrinkage ->
# more genomic variance allowed. If ICC_site still dominates ICC_genomic
# as p0 grows, the conclusion isn't an artifact of over-shrinking.
def icc_means(tr, Xb):
    post = tr.posterior
    b  = post["beta"].stack(s=("chain", "draw")).values        # (p, S)
    a  = post["alpha"].stack(s=("chain", "draw")).values       # (S,)
    ss = post["sigma_site"].stack(s=("chain", "draw")).values  # (S,)
    vg = (Xb @ b).var(axis=0)
    vs = ss ** 2
    vr = np.pi**2 / (6 * a**2)
    tot = vs + vg + vr
    return (vs / tot).mean(), (vg / tot).mean()

Xb = df[df["OS_time"] > 0][gene_cols].values
rows = []
for p0 in [10, 50, 100]:
    mb = build_model_b(df, gene_cols, p0=p0)
    with mb:
        tr = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.99,
                       nuts_sampler="nutpie", return_inferencedata=True, progressbar=False)
    icc_site, icc_gen = icc_means(tr, Xb)
    bs = az.summary(tr, var_names=["beta"]).reset_index(drop=True)
    bs[["eti89_lb", "eti89_ub"]] = bs[["eti89_lb", "eti89_ub"]].apply(pd.to_numeric, errors="coerce")
    n_hits = int(((bs["eti89_lb"] > 0) | (bs["eti89_ub"] < 0)).sum())
    rows.append({"p0": p0, "ICC_site": icc_site, "ICC_genomic": icc_gen,
                 "sigma_site": float(tr.posterior["sigma_site"].mean()),
                 "gene_hits": n_hits, "divergences": int(tr.sample_stats.diverging.sum())})
    print(f"p0={p0:3d}: ICC_site={icc_site:.3f}  ICC_genomic={icc_gen:.3f}  hits={n_hits}")

p0_sens = pd.DataFrame(rows)
print()
print(p0_sens.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

## Leave-one-site-out C-index (predictive generalisation)

The ICC says how much variance sits at the site level; this asks the complementary **predictive** question (plan §3.5): *do site-adjusted genomic coefficients generalise to **unseen** sites better than naive ones?*

For each TSS site: fit Model A and Model B on all *other* sites, then predict a genomic risk score η = β·x on the held-out site. Model B's site intercept is unidentified for a brand-new site, so it is set to its prior mean (0) — i.e. we score **genomic discrimination only**, which is the fair test of transfer. Out-of-fold predictions are pooled across all sites and scored with Harrell's C-index; a higher C-index for Model B means site-adjusted training produced coefficients that transfer better to new institutions.

This is **pure leave-one-site-out** (each site its own fold → pooled C-index, no fold-balancing needed). It re-fits both models once per site, so it is heavy — CV draws are reduced (only the posterior-mean β is needed). **Expect ~45–60 min.** Lower `CV_DRAWS`/`CV_CHAINS` to trade precision for speed, or lift this exact loop into the server script.

In [ ]:
# --- Leave-one-site-out C-index (predictive generalisation to unseen sites) ---
# Re-fits Model A and Model B holding out each TSS site, predicts a genomic
# risk score eta = beta.x on the held-out site, pools out-of-fold predictions,
# and scores with Harrell's C-index. HEAVY: ~2 * n_sites model fits.
dfl   = df[df["OS_time"] > 0].reset_index(drop=True)
Xall  = dfl[gene_cols].values
t_all = dfl["OS_time"].values
e_all = dfl["OS"].values.astype(int)
sites = dfl["tss"].unique()

CV_DRAWS, CV_TUNE, CV_CHAINS = 500, 500, 2     # posterior-mean beta only; reduce/raise as needed

def fit_mean_beta(build_fn, train_df):
    with build_fn(train_df, gene_cols, p0=10):
        tr = pm.sample(draws=CV_DRAWS, tune=CV_TUNE, chains=CV_CHAINS,
                       target_accept=0.99, nuts_sampler="nutpie",
                       return_inferencedata=True, progressbar=False)
    return tr.posterior["beta"].mean(("chain", "draw")).values   # (p,)

eta_A = np.full(len(dfl), np.nan)
eta_B = np.full(len(dfl), np.nan)
for k, s in enumerate(sites, 1):
    test = dfl["tss"].values == s
    train_df = dfl[~test]
    eta_A[test] = Xall[test] @ fit_mean_beta(build_model_a, train_df)
    eta_B[test] = Xall[test] @ fit_mean_beta(build_model_b, train_df)   # unseen-site intercept = 0
    print(f"[{k:2d}/{len(sites)}] site {s}: {int(test.sum()):3d} held out")

def c_index(times, events, scores):
    # Harrell's C: higher score = predicted longer survival. Comparable pairs
    # are those where the earlier event time is an observed death.
    num = den = 0.0
    for i in np.where(events == 1)[0]:
        later = times > times[i]
        den  += later.sum()
        num  += (scores[later] > scores[i]).sum() + 0.5 * (scores[later] == scores[i]).sum()
    return num / den if den else np.nan

cA, cB = c_index(t_all, e_all, eta_A), c_index(t_all, e_all, eta_B)

# bootstrap 89% CI on the gap (resample patients)
rng, gaps = np.random.default_rng(0), []
for _ in range(1000):
    idx = rng.integers(0, len(dfl), len(dfl))
    gaps.append(c_index(t_all[idx], e_all[idx], eta_B[idx]) - c_index(t_all[idx], e_all[idx], eta_A[idx]))
lo, hi = np.nanpercentile(gaps, [5.5, 94.5])

print(f"\nLOSO pooled C-index  Model A (naive)    : {cA:.4f}")
print(f"LOSO pooled C-index  Model B (site-adj) : {cB:.4f}")
print(f"gap (B - A): {cB - cA:+.4f}   89% bootstrap CI [{lo:+.4f}, {hi:+.4f}]")

## Conclusion — LUSC

**RQ1 (variance attribution): site is the dominant *explainable* source of survival variation.**
From the site-adjusted model (Model B, 0 divergences, all r-hat ≈ 1.00):

| component | ICC (mean) | 89% ETI |
|---|---|---|
| **site** | **9.9%** | **3.2 – 19.7%** |
| genomic (hallmark) | 1.5% | 0.0 – 4.9% |
| residual | 88.6% | 78.6 – 96.0% |

- `sigma_site = 0.457` (89% ETI 0.26–0.69) — tissue source site contributes a **credibly non-zero** share of survival-outcome variance (ICC_site lower bound 3.2% excludes ~0).
- **Site explains ≈ 6–7× the variance of the entire hallmark genomic panel** (ICC_site 9.9% vs ICC_genomic 1.5%). Ignoring the irreducible residual, **~87% of the *explainable* survival variance in LUSC is institutional, not biological** — direct evidence of the site confounding this study targets.

**RQ3 (gene attenuation): null for LUSC — and that is itself informative.**
After regularized-horseshoe shrinkage, **0 / 3,867 hallmark genes** have an 89% ETI excluding 0 in *either* model. With no individual-gene OS signal surviving shrinkage, there is nothing to "attenuate" under site adjustment in this cohort. **Robustness confirmed:** across `p0 ∈ {10, 50, 100}`, ICC_site stays ≈ 0.10 while ICC_genomic stays ≤ 0.02 (a ~5–6× gap) and gene-hits remain 0 — the site ≫ genomics conclusion is not an artifact of the shrinkage prior.

**Predictive generalisation (leave-one-site-out).** Both models reach **C-index ≈ 0.53** on held-out sites (A 0.531, B 0.534; gap **+0.003**, 89% bootstrap CI **[−0.006, +0.011]**). The hallmark panel does **not** discriminate OS out-of-sample, and site adjustment neither helps nor hurts genomic transfer — as expected given the null gene signal. This confirms the RQ1 site effect operates as a per-institution **baseline shift** (the intercepts `μ_s`), *not* by inflating gene effects; so in LUSC there is no gene-level "leakage" to expose (there was no gene signal to begin with). Consistent with the published literature: optimised LUSC RNA-seq signatures report only modest discrimination (~0.62–0.67) under outcome-driven selection and non-site-preserved CV, and a pre-specified panel under honest leave-one-site-out evaluation drops toward chance.

**Caveats.** (i) Single cohort — the ~10% ICC_site is LUSC-specific and is exactly the quantity RQ2 compares across cancer types. (ii) The residual dominates by construction (`σ²_residual = π²/6α²`); the meaningful result is the site-vs-genomic contrast, not the absolute residual share. (iii) ICC_genomic is small partly because the horseshoe shrinks hard — the `p0` sweep above shows the conclusion survives loosening it. (iv) At `p0=100` a few divergences appear (9/4000 ≈ 0.2%) as the looser prior sharpens the funnel; immaterial to the stable ICC estimate but would warrant higher `target_accept` if `p0=100` were used as primary.

# Robustness & sensitivity

Three checks on the primary result. **The hallmark panel on the primary endpoint stays the reported model** — everything below produces *robustness rows*, not a replacement.

1. **PFI as primary endpoint** — Liu et al. (2018) recommend PFI over OS for most cancer types (more events, less confounded by non-cancer death). We report **PFI primary, OS secondary**. *Note: PFI has more events than OS, so the gene null and the ICC may differ from the OS analysis above — this is a genuine re-analysis, not relabeling.*
2. **Top-variance panel** — data-driven gene selection (top-N by variance, no outcome peeking). If the gene null persists here, it is **not** an artifact of restricting to hallmark. This is a sensitivity check only — the hallmark panel remains the headline.
3. **ComBat** — batch-correct expression by TSS, then refit. ComBat removes site variance **by construction** (`sigma_site → ~0`) and cannot decompose site vs. biology the way the random-effect model does — demonstrating empirically why we model site rather than ComBat it away.

Each `run_config` call fits Model A **and** Model B (~2 nutpie fits); the full section is ~12 fits (~30 min). Run when you want the robustness suite.

In [ ]:
# --- robustness helpers (reuse the now endpoint-parametrized build_model_a/b) ---
from inmoose.pycombat import pycombat_norm

def _valid_ep(d, ep):
    tc, ec = f"{ep}_time", ep
    return d[(d[tc] > 0) & d[ec].notna() & d[tc].notna()]

def combat_by_tss(d, cols):
    """ComBat-correct the gene block by TSS; drop singleton sites (ComBat needs >=2/batch)."""
    vc = d["tss"].value_counts()
    sub = d[d["tss"].isin(vc[vc >= 2].index)].copy()
    sub[cols] = pycombat_norm(sub[cols].T.values, sub["tss"].values).T
    return sub

def _hits(tr):
    b = tr.posterior["beta"].stack(s=("chain", "draw")).values
    return int(((np.percentile(b, 5.5, 1) > 0) | (np.percentile(b, 94.5, 1) < 0)).sum())

def run_config(df_in, cols, endpoint="OS", combat=False, p0=10,
               draws=1000, tune=1000, chains=4, target_accept=0.99):
    """Fit Model A + B for one (endpoint, panel, combat) config; return ICC + hits."""
    tc, ec = f"{endpoint}_time", endpoint
    d = combat_by_tss(df_in, cols) if combat else df_in
    dv = _valid_ep(d, endpoint)
    Xb = dv[cols].values
    with build_model_a(d, cols, p0=p0, time_col=tc, event_col=ec):
        ta = pm.sample(draws=draws, tune=tune, chains=chains, target_accept=target_accept,
                       nuts_sampler="nutpie", return_inferencedata=True, progressbar=False)
    with build_model_b(d, cols, p0=p0, time_col=tc, event_col=ec):
        tb = pm.sample(draws=draws, tune=tune, chains=chains, target_accept=target_accept,
                       nuts_sampler="nutpie", return_inferencedata=True, progressbar=False)
    bs = tb.posterior["beta"].stack(s=("chain", "draw")).values
    al = tb.posterior["alpha"].stack(s=("chain", "draw")).values
    sg = tb.posterior["sigma_site"].stack(s=("chain", "draw")).values
    vg, vs, vr = (Xb @ bs).var(0), sg**2, np.pi**2 / (6 * al**2)
    tot = vs + vg + vr
    return {"endpoint": endpoint, "combat": combat, "n": len(dv), "events": int(dv[ec].sum()),
            "genes": len(cols), "ICC_site": float((vs / tot).mean()),
            "ICC_genomic": float((vg / tot).mean()), "sigma_site": float(sg.mean()),
            "hits_A": _hits(ta), "hits_B": _hits(tb), "div_B": int(tb.sample_stats.diverging.sum())}

In [ ]:
# 1. PFI as PRIMARY endpoint (Liu et al.), hallmark panel; OS kept as secondary
robust = []
for ep in ["PFI", "OS"]:
    r = run_config(df, gene_cols, endpoint=ep); r["panel"] = "hallmark"
    robust.append(r); print(ep, "->", r)
pd.DataFrame(robust)

In [ ]:
# 2. top-variance panel robustness (data-driven; NOT the reported model)
#    if the gene null persists here, it is not an artifact of the hallmark restriction.
ensg_all = [c for c in df.columns if c.startswith("ENSG")]
for N in [500, 2000]:
    cols = df[ensg_all].var().nlargest(N).index.tolist()
    r = run_config(df, cols, endpoint="PFI"); r["panel"] = f"topvar:{N}"
    robust.append(r); print(f"topvar:{N} (PFI) ->", r)
pd.DataFrame(robust)

In [ ]:
# 3. ComBat sensitivity: batch-correct expression by TSS, then refit (hallmark, PFI)
raw = run_config(df, gene_cols, endpoint="PFI", combat=False)
cbt = run_config(df, gene_cols, endpoint="PFI", combat=True)
print("raw    :", raw)
print("combat :", cbt)
print(f"\nsigma_site  raw={raw['sigma_site']:.3f}  ->  combat={cbt['sigma_site']:.3f}")
print("ComBat removes the site variance BY CONSTRUCTION (sigma_site -> ~0); the "
      "random-effect model instead PARTITIONS it (ICC_site) without discarding "
      "site-correlated biology — which is why we model site rather than ComBat it away.")